# Formative Assignment: Advanced Linear Algebra (PCA)

## Kenya Malaria HDX Dataset

This notebook completes the PCA formative assignment using Kenyan malaria-related county data from the Humanitarian Data Exchange (HDX). The data is Africanized because it focuses on Kenya counties and malaria/fever burden, bed-net use, and malaria-related health spending.

Important constraints followed:
1. PCA is implemented from scratch using NumPy.
2. No sklearn is used.
3. CSV loading uses Python standard library tools only.
4. Matplotlib is used only for the required visualizations.
5. Missing values and non-numeric columns are identified and handled before PCA.


## Step 0: Load and Merge HDX Kenya Malaria Data

The HDX resources are small county-level CSV files. To create a stronger PCA dataset with at least seven columns, the notebook merges three HDX Kenya malaria resources by county: malaria cases per 100,000 people, bed-net and fever/malaria occurrence, and malaria-related spending per county.


In [ ]:
import numpy as np
import csv
import io
import os
import time
import urllib.request
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)


In [ ]:
# HDX resource URLs and local fallbacks.
DATASETS = {
    "cases": {
        "url": "https://data.humdata.org/dataset/dae1bcde-0238-43d5-ab16-e0726e4ecaa5/resource/55e8e082-8fa1-4d47-a8e3-f6d2bcbe3c6a/download/malaria-cases-per-100000-people-in-kenya-per-county.csv",
        "local": "kenya_malaria_cases_per_100000_county.csv"
    },
    "spending": {
        "url": "https://data.humdata.org/dataset/2ab7d812-be62-48cb-bd94-407fd522e21e/resource/90f49a6d-9f27-4b5d-aad3-2fc842f6e5cc/download/county_malaria_rates_vs_spending.csv",
        "local": "kenya_malaria_spending_county.csv"
    },
    "bednets": {
        "url": "https://data.humdata.org/dataset/2273977c-2ca6-4aaf-a76c-04caa16d6be3/resource/01e48f2a-f712-4251-bb9e-11ca4df3c4a7/download/bed_nets_and_illness_by_county.csv",
        "local": "kenya_malaria_bednets_county.csv"
    }
}

missing_markers = {"", "NA", "N/A", "NaN", "nan", "None", "null", "NULL"}

def read_hdx_csv(resource):
    url = resource["url"]
    local_file = resource["local"]
    try:
        with urllib.request.urlopen(url, timeout=30) as response:
            text = response.read().decode("utf-8-sig")
        print("Loaded online:", url)
    except Exception as exc:
        print("Online load failed, using local file:", local_file, "|", exc)
        with open(local_file, "r", encoding="utf-8-sig") as f:
            text = f.read()
    rows = list(csv.DictReader(io.StringIO(text)))
    return rows

raw_cases = read_hdx_csv(DATASETS["cases"])
raw_spending = read_hdx_csv(DATASETS["spending"])
raw_bednets = read_hdx_csv(DATASETS["bednets"])

print("Rows in cases resource:", len(raw_cases))
print("Rows in spending resource:", len(raw_spending))
print("Rows in bed-net resource:", len(raw_bednets))
print("Cases columns:", list(raw_cases[0].keys()))
print("Spending columns:", list(raw_spending[0].keys()))
print("Bed-net columns:", list(raw_bednets[0].keys()))


In [ ]:
# Check original missing values before cleaning.
def count_missing_values(rows):
    total = 0
    by_column = {}
    if not rows:
        return total, by_column
    for col in rows[0].keys():
        count = 0
        for row in rows:
            if str(row.get(col, "")).strip() in missing_markers:
                count += 1
        by_column[col] = count
        total += count
    return total, by_column

for name, rows in [("cases", raw_cases), ("spending", raw_spending), ("bednets", raw_bednets)]:
    total_missing, missing_by_col = count_missing_values(rows)
    print(name, "total missing values:", total_missing)
    print(missing_by_col)


In [ ]:
# Helper functions for cleaning and merging.
def parse_number(value):
    value = str(value).strip().replace(",", "")
    if value in missing_markers:
        return np.nan
    try:
        return float(value)
    except ValueError:
        return np.nan

def clean_text(value):
    value = str(value).strip()
    return "Unknown" if value in missing_markers else value

def normalize_county(value):
    value = clean_text(value).upper().replace("'", "")
    chars = []
    for ch in value:
        if ch.isalnum():
            chars.append(ch)
    key = "".join(chars)
    return key if key else "UNKNOWN"

def display_county(value):
    value = clean_text(value)
    if value == "Unknown":
        return value
    return " ".join(value.replace("-", " ").split()).title()

def ensure_record(records, county_value, country="Kenya"):
    key = normalize_county(county_value)
    if key not in records:
        records[key] = {
            "country": clean_text(country),
            "county": display_county(county_value),
            "malaria_cases_per_100000": np.nan,
            "spending_objectid": np.nan,
            "slept_under_bed_net_percent_spending": np.nan,
            "had_fever_or_malaria_percent_spending": np.nan,
            "spending_per_person_ksh": np.nan,
            "spending_per_person_ksh_duplicate": np.nan,
            "bednets_objectid": np.nan,
            "slept_under_bed_net_percent_bednets": np.nan,
            "had_fever_or_malaria_percent_bednets": np.nan,
            "health_spending_per_person_score": np.nan
        }
    return records[key]

records = {}

# Cases resource includes a national average row and one blank row. The national aggregate is not a county.
for row in raw_cases:
    county = clean_text(row.get("Counties", ""))
    if county.upper() == "NATIONAL AVERAGE":
        continue
    rec = ensure_record(records, county, clean_text(row.get("Country", "Kenya")))
    rec["malaria_cases_per_100000"] = parse_number(row.get("Malaria cases (per 100,000 people)", ""))

for row in raw_spending:
    rec = ensure_record(records, row.get("county", ""), "Kenya")
    rec["spending_objectid"] = parse_number(row.get("objectid", ""))
    rec["slept_under_bed_net_percent_spending"] = parse_number(row.get("slept_under_a_bed_net_in_percen", ""))
    rec["had_fever_or_malaria_percent_spending"] = parse_number(row.get("had_fever_or_malaria_in_percent", ""))
    rec["spending_per_person_ksh"] = parse_number(row.get("spending_per_person_in_ksh", ""))
    rec["spending_per_person_ksh_duplicate"] = parse_number(row.get("spending_per_person_in_ksh_1", ""))

for row in raw_bednets:
    rec = ensure_record(records, row.get("name_of_county", ""), "Kenya")
    rec["bednets_objectid"] = parse_number(row.get("objectid", ""))
    rec["slept_under_bed_net_percent_bednets"] = parse_number(row.get("percentagethat_slept_under_a_be", ""))
    rec["had_fever_or_malaria_percent_bednets"] = parse_number(row.get("percentage_that_had_a_fever_or_", ""))
    rec["health_spending_per_person_score"] = parse_number(row.get("heath_spending_per_person", ""))

merged_rows = list(records.values())
print("Merged rows:", len(merged_rows))
print("First 5 merged records:")
for row in merged_rows[:5]:
    print(row)


In [ ]:
# Add transparent, use-case-derived features for PCA.
def malaria_burden_group(cases):
    if np.isnan(cases):
        return "Unknown"
    if cases >= 40000:
        return "High burden"
    if cases >= 20000:
        return "Medium burden"
    return "Lower burden"

for row in merged_rows:
    bednet = row["slept_under_bed_net_percent_spending"]
    fever = row["had_fever_or_malaria_percent_spending"]
    spending = row["spending_per_person_ksh"]
    cases = row["malaria_cases_per_100000"]

    row["malaria_burden_group"] = malaria_burden_group(cases)
    row["county_name_length"] = len(row["county"])
    row["prevention_gap_percent"] = fever - bednet if not (np.isnan(fever) or np.isnan(bednet)) else np.nan
    row["spending_per_1000_cases"] = spending / (cases / 1000) if not (np.isnan(spending) or np.isnan(cases) or cases == 0) else np.nan

all_columns = list(merged_rows[0].keys())
print("Columns in merged/engineered dataset:", len(all_columns))
print(all_columns)
print("Example engineered rows:")
for row in merged_rows[:5]:
    print(row["county"], row["malaria_burden_group"], row["prevention_gap_percent"], row["spending_per_1000_cases"])


In [ ]:
# Prepare numeric and categorical data.
# County is a non-numeric identifier, so it is retained for labels rather than treated as an ordered PCA variable.
identifier_cols = ["county"]
categorical_cols = ["country", "malaria_burden_group"]
numeric_cols = [
    "malaria_cases_per_100000",
    "spending_objectid",
    "slept_under_bed_net_percent_spending",
    "had_fever_or_malaria_percent_spending",
    "spending_per_person_ksh",
    "spending_per_person_ksh_duplicate",
    "bednets_objectid",
    "slept_under_bed_net_percent_bednets",
    "had_fever_or_malaria_percent_bednets",
    "health_spending_per_person_score",
    "county_name_length",
    "prevention_gap_percent",
    "spending_per_1000_cases"
]

county_labels = np.array([row["county"] for row in merged_rows])
X_numeric = np.array([[row[col] for col in numeric_cols] for row in merged_rows], dtype=float)
missing_numeric_before = np.isnan(X_numeric).sum()

# Median imputation for missing numeric values.
col_medians = np.nanmedian(X_numeric, axis=0)
missing_positions = np.where(np.isnan(X_numeric))
X_numeric[missing_positions] = np.take(col_medians, missing_positions[1])

print("Numeric columns:", len(numeric_cols))
print("Missing numeric values before imputation:", missing_numeric_before)
print("Missing numeric values after imputation:", np.isnan(X_numeric).sum())


In [ ]:
# One-hot encode categorical columns using NumPy.
def one_hot_encode(values, prefix):
    cleaned = np.array([clean_text(v) for v in values])
    unique_values, inverse = np.unique(cleaned, return_inverse=True)
    encoded = np.eye(len(unique_values))[inverse]
    names = np.array([prefix + "=" + value for value in unique_values])
    return encoded, names, unique_values

encoded_parts = []
encoded_feature_names = []
category_maps = {}
for col in categorical_cols:
    encoded, names, unique_values = one_hot_encode([row[col] for row in merged_rows], col)
    encoded_parts.append(encoded)
    encoded_feature_names.extend(names.tolist())
    category_maps[col] = unique_values

X_categorical = np.column_stack(encoded_parts)
raw_data = np.column_stack([X_numeric, X_categorical])
feature_names = np.array(numeric_cols + encoded_feature_names)

print("Categorical maps:")
for col, values in category_maps.items():
    print(col, "->", values)
print("Final PCA feature matrix shape:", raw_data.shape)
print("First 5 rows, first 8 features:")
print(raw_data[:5, :8])


### Step 1: Load and Standardize the Data

Before applying PCA, we standardize the dataset. Standardization makes every feature have mean 0 and standard deviation 1, so large-scale variables such as spending in KSH do not dominate percentage variables such as bed-net use.


In [ ]:
# Step 1: Standardize the data using the formula z = (x - mean) / standard deviation.
feature_means = np.mean(raw_data, axis=0)
feature_stds = np.std(raw_data, axis=0, ddof=0)
feature_stds[feature_stds == 0] = 1
standardized_data = (raw_data - feature_means) / feature_stds

print("Standardized data shape:", standardized_data.shape)
print("Means after standardization, first 8 features:")
print(np.mean(standardized_data, axis=0)[:8])
print("Standard deviations after standardization, first 8 features:")
print(np.std(standardized_data, axis=0, ddof=0)[:8])
standardized_data[:5]


### Step 3: Calculate the Covariance Matrix

The covariance matrix helps us understand how features vary together. It is the matrix that PCA decomposes to find principal directions of variance.


In [ ]:
# Step 3: Calculate the covariance matrix from scratch.
n_samples = standardized_data.shape[0]
cov_matrix = (standardized_data.T @ standardized_data) / (n_samples - 1)

print("Covariance matrix shape:", cov_matrix.shape)
cov_matrix[:8, :8]


We compute the covariance matrix because PCA needs to know which malaria-related features move together across Kenyan counties. First, it captures relationships such as whether high fever/malaria occurrence aligns with high reported malaria cases or spending. Second, its eigenvectors define the new principal component axes that summarize the strongest combined patterns in the county data.


### Step 4: Perform Eigendecomposition

Eigendecomposition of the covariance matrix gives eigenvalues and eigenvectors. Eigenvalues measure the amount of variance explained by each principal component, while eigenvectors give the component directions.


In [ ]:
# Step 4: Perform eigendecomposition.
# The covariance matrix is symmetric, so np.linalg.eigh is appropriate and stable.
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

print("Eigenvalues before sorting, first 10:")
print(eigenvalues[:10])
print("Eigenvector matrix shape:", eigenvectors.shape)
eigenvalues, eigenvectors


### Step 5: Sort Principal Components

The eigenvectors are sorted by descending eigenvalue so that PC1 explains the most variance, PC2 explains the next most variance, and so on.


In [ ]:
# Step 5: Sort principal components.
sorted_indices = np.argsort(eigenvalues)[::-1]
sorted_eigenvalues = eigenvalues[sorted_indices]
sorted_eigenvectors = eigenvectors[:, sorted_indices]

explained_variance_ratio = sorted_eigenvalues / np.sum(sorted_eigenvalues)
cumulative_explained_variance = np.cumsum(explained_variance_ratio)

print("Top 10 eigenvalues:")
print(sorted_eigenvalues[:10])
print("Top 10 explained variance percentages:")
print((explained_variance_ratio[:10] * 100).round(2))
print("Top 10 cumulative explained variance percentages:")
print((cumulative_explained_variance[:10] * 100).round(2))
sorted_eigenvectors[:8, :5]


### Task 2: Dynamically Select Principal Components

The number of components is selected dynamically using a cumulative explained variance threshold. A 90% threshold is used to preserve most of the malaria-related information while reducing the number of dimensions.


In [ ]:
# Task 2: Dynamically select the number of principal components.
variance_threshold = 0.90
num_components = int(np.searchsorted(cumulative_explained_variance, variance_threshold) + 1)

print("Variance threshold:", variance_threshold)
print("Selected number of principal components:", num_components)
print("Explained variance retained:", round(cumulative_explained_variance[num_components - 1] * 100, 2), "%")
print("Original number of PCA features:", standardized_data.shape[1])
print("Reduced number of PCA features:", num_components)


### Step 6: Project Data onto Principal Components

Now the standardized data is multiplied by the selected eigenvectors to transform counties into principal-component space.


In [ ]:
# Step 6: Project data onto the selected principal components.
selected_components = sorted_eigenvectors[:, :num_components]
reduced_data = standardized_data @ selected_components

print("Reduced data shape:", reduced_data.shape)
reduced_data[:5]


### Step 7: Output the Reduced Data

The reduced data is the PCA-transformed representation of each county observation.


In [ ]:
# Step 7: Output the reduced data.
print(f"Reduced Data Shape: {reduced_data.shape}")
print("First five counties and their first principal component values:")
for county, values in zip(county_labels[:5], reduced_data[:5]):
    print(county, values[:min(5, len(values))])
reduced_data[:5]


### Task 3: Optimize PCA Implementation and Benchmarking

The PCA implementation below is vectorized with NumPy. It avoids loops during standardization, covariance computation, eigendecomposition, sorting, threshold selection, and projection.


In [ ]:
def pca_numpy(X, threshold=0.90):
    means = np.mean(X, axis=0)
    stds = np.std(X, axis=0, ddof=0)
    stds[stds == 0] = 1
    Z = (X - means) / stds
    C = (Z.T @ Z) / (Z.shape[0] - 1)
    vals, vecs = np.linalg.eigh(C)
    order = np.argsort(vals)[::-1]
    vals = vals[order]
    vecs = vecs[:, order]
    ratios = vals / np.sum(vals)
    cumulative = np.cumsum(ratios)
    k = int(np.searchsorted(cumulative, threshold) + 1)
    projected = Z @ vecs[:, :k]
    return projected, vals, vecs, ratios, cumulative, k

start = time.perf_counter()
projected_once, vals_once, vecs_once, ratios_once, cumulative_once, k_once = pca_numpy(raw_data, variance_threshold)
elapsed = time.perf_counter() - start

large_data = np.tile(raw_data, (1000, 1))
start_large = time.perf_counter()
projected_large, _, _, _, cumulative_large, k_large = pca_numpy(large_data, variance_threshold)
elapsed_large = time.perf_counter() - start_large

print("Benchmark on HDX Kenya malaria dataset:")
print("Rows:", raw_data.shape[0], "Features:", raw_data.shape[1], "Components:", k_once, "Seconds:", round(elapsed, 6))
print("Benchmark on repeated larger dataset:")
print("Rows:", large_data.shape[0], "Features:", large_data.shape[1], "Components:", k_large, "Seconds:", round(elapsed_large, 6))


### Step 8: Visualize Before and After PCA

The first plot compares the original malaria-related feature space. The second plot shows the same county observations after PCA using PC1 and PC2.


In [ ]:
# Step 8: Visualize before and after PCA.
feature_x = "slept_under_bed_net_percent_spending"
feature_y = "had_fever_or_malaria_percent_spending"
x_idx = int(np.where(feature_names == feature_x)[0][0])
y_idx = int(np.where(feature_names == feature_y)[0][0])
case_idx = int(np.where(feature_names == "malaria_cases_per_100000")[0][0])
color_values = X_numeric[:, numeric_cols.index("malaria_cases_per_100000")]

pc2_values = reduced_data[:, 1] if reduced_data.shape[1] > 1 else np.zeros(reduced_data.shape[0])

plt.figure(figsize=(13, 5))

plt.subplot(1, 2, 1)
plt.scatter(standardized_data[:, x_idx], standardized_data[:, y_idx], c=color_values, cmap="plasma", alpha=0.82, edgecolor="black", linewidth=0.35)
plt.xlabel("Bed-net use percent (standardized)")
plt.ylabel("Fever or malaria percent (standardized)")
plt.title("Before PCA: Original Feature Space")
plt.grid(alpha=0.25)

plt.subplot(1, 2, 2)
plt.scatter(reduced_data[:, 0], pc2_values, c=color_values, cmap="plasma", alpha=0.82, edgecolor="black", linewidth=0.35)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("After PCA: Principal Component Space")
plt.grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:
# Explained variance plot for component selection.
plt.figure(figsize=(8, 5))
component_numbers = np.arange(1, len(cumulative_explained_variance) + 1)
plt.plot(component_numbers, cumulative_explained_variance * 100, marker="o", linewidth=1.5)
plt.axhline(variance_threshold * 100, color="red", linestyle="--", label="90% threshold")
plt.axvline(num_components, color="gray", linestyle=":", label=f"Selected k = {num_components}")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance (%)")
plt.title("Dynamic Principal Component Selection")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


### Interpretation of the Before and After PCA Visuals

The before-PCA plot only compares bed-net use with fever/malaria occurrence, so it shows one direct public-health relationship. The after-PCA plot uses all cleaned features together, so PC1 and PC2 summarize broader county patterns across malaria cases, fever/malaria occurrence, bed-net use, and spending. The counties are the same observations, but they are rotated into a space where the first component has the highest variance.


### Why This Number of Principal Components Was Selected

The selected number of components is the smallest value that keeps at least 90% of the total explained variance. This reduces the dataset from many original and encoded features into fewer dimensions while keeping most information. The tradeoff is interpretability and detail: fewer components simplify comparison, but smaller county-specific differences may be pushed into discarded components.


### What Information Is Lost When Reducing Dimensions?

For this Kenyan malaria use case, reduction can lose detailed differences between specific indicators such as exact malaria cases, bed-net coverage, fever/malaria percentage, and county spending. PCA keeps broad patterns like malaria burden, prevention coverage, and spending pressure, but it does not preserve every original feature separately. Some local county-level detail is compressed when later components are removed.


## References

- HDX: Malaria cases per 100,000 people in Kenya, https://data.humdata.org/dataset/malaria-cases-per-100-000-people-in-kenya
- HDX: Kenya - Expenditure on Malaria Per capita per county, https://data.humdata.org/dataset/kenya-health-spending-per-capita-per-county
- HDX: Kenya - Bed Nets, Malaria and Fever occurrence and Health spending per County, https://data.humdata.org/dataset/bed-nets-and-illness-by-county-kenya
